# Download Large Files

Downloads `files.zip` from Google Drive, extracts it to the project root, then deletes the zip.

**Contents of files.zip:**
- `Leo/Other Outputs/arxiv_embeddings.npz`
- `Xiong's output/data/arxiv-metadata-oai-snapshot.json`
- `Xiong's output/data/citation_results.jsonl`
- `Xiong's output/outputs/arxiv_ids_all.csv`
- `Xiong's output/outputs/arxiv_metadata_clean.parquet`
- `Xiong's output/outputs/arxiv_with_citations.parquet`

In [ ]:
%pip install -q gdown tqdm

In [ ]:
import platform, os
from pathlib import Path

OS = platform.system()  # 'Windows' or 'Darwin' (macOS)
ROOT = Path(os.path.dirname(os.path.abspath("__file__")))
ZIP_PATH = ROOT / "files.zip"
FILE_ID = "1QPbh5S4w9RDdgy4YRCZO0kKMp9fBgv2f"

print(f"OS      : {OS}")
print(f"Project : {ROOT}")
print(f"Zip out : {ZIP_PATH}")

## Step 1 — Download files.zip

In [ ]:
import requests
from tqdm.notebook import tqdm

def download_gdrive_with_progress(file_id: str, dest: Path):
    """Download a Google Drive file with a tqdm progress bar.
    Handles large-file virus-scan confirmation pages automatically.
    """
    gdrive_url = f"https://drive.google.com/uc?id={file_id}"
    session = requests.Session()

    # First request — may redirect to confirmation page for large files
    resp = session.get(gdrive_url, stream=True, timeout=60)

    # If Google asks for confirmation (large file warning), follow it
    for key, value in resp.cookies.items():
        if key.startswith("download_warning"):
            params = {"id": file_id, "confirm": value, "export": "download"}
            resp = session.get("https://drive.google.com/uc", params=params,
                               stream=True, timeout=60)
            break

    total = int(resp.headers.get("Content-Length", 0)) or None

    with open(dest, "wb") as f, tqdm(
        desc="files.zip",
        total=total,
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
        dynamic_ncols=True,
    ) as bar:
        for chunk in resp.iter_content(chunk_size=1024 * 1024):  # 1 MB chunks
            if chunk:
                f.write(chunk)
                bar.update(len(chunk))

if ZIP_PATH.exists():
    print(f"[SKIP] files.zip already exists ({ZIP_PATH.stat().st_size / 1024**2:.0f} MB). Delete it to re-download.")
else:
    print("Starting download...")
    download_gdrive_with_progress(FILE_ID, ZIP_PATH)
    print(f"\nDownloaded: {ZIP_PATH.stat().st_size / 1024**2:.1f} MB")

## Step 2 — Extract and clean up

In [ ]:
import zipfile
from tqdm.notebook import tqdm

if not ZIP_PATH.exists():
    raise FileNotFoundError(f"{ZIP_PATH.name} not found — run Step 1 first.")

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    members = zf.infolist()

    # macOS: strip __MACOSX metadata entries
    if OS == "Darwin":
        members = [m for m in members if not m.filename.startswith("__MACOSX")]

    total_bytes = sum(m.file_size for m in members)
    print(f"Extracting {len(members)} entries ({total_bytes / 1024**2:.0f} MB uncompressed) to:")
    print(f"  {ROOT}")

    with tqdm(total=total_bytes, unit="B", unit_scale=True,
              unit_divisor=1024, dynamic_ncols=True, desc="Extracting") as bar:
        for member in members:
            zf.extract(member, ROOT)
            bar.update(member.file_size)

print("Extraction complete.")

ZIP_PATH.unlink()
print(f"Deleted files.zip")

## Step 3 — Verify

In [ ]:
EXPECTED = [
    "Leo/Other Outputs/arxiv_embeddings.npz",
    "Xiong's output/data/arxiv-metadata-oai-snapshot.json",
    "Xiong's output/data/citation_results.jsonl",
    "Xiong's output/outputs/arxiv_ids_all.csv",
    "Xiong's output/outputs/arxiv_metadata_clean.parquet",
    "Xiong's output/outputs/arxiv_with_citations.parquet",
]

print("Verification:")
all_ok = True
for rel in EXPECTED:
    p = ROOT / rel
    if p.exists():
        print(f"  OK    {rel}  ({p.stat().st_size / 1024**2:.1f} MB)")
    else:
        print(f"  FAIL  {rel}  — NOT FOUND")
        all_ok = False

print("\nAll files present!" if all_ok else "\nSome files are missing — check the zip structure.")